# Live Inference Relay — running on AMD Developer Cloud

This notebook runs on this AMD Jupyter instance and exposes a tiny local HTTP server
(`POST /v1/chat/completions`) that `backend/agent_core.py` calls for live specialist/
synthesis inference. It handles two kinds of requests based on the `model` field:

- `model` is `qwen2.5-coder:7b` or `gemma2` → runs that model **locally via Ollama on
  this AMD instance's GPU** — genuine on-device compute, selected from the frontend's
  provider switcher ("AMD Notebook — Qwen Coder" / "AMD Notebook — Gemma 2"). Qwen2.5-
  Coder replaces plain Llama 3 here because every specialist in this pipeline asks the
  LLM to WRITE PYTHON CODE that then gets executed - a code-specialized model produces
  far fewer execution-time errors than a general chat model.
- any other `model` value → forwarded to `api.featherless.ai` from this notebook
  process, dispatched from the AMD box (used for the Featherless fallback route).

Run cells top to bottom, then leave this notebook open/running while you use the backend.

**Run the device-info cell first and keep its output in the committed notebook** — same
convention as `specialist_eval_and_embeddings.ipynb`.

In [ ]:
# --- Device info: confirm this notebook process is actually on the AMD box ---
import subprocess

try:
    print(subprocess.run(["rocm-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("rocm-smi not found on PATH (fine if you're smoke-testing this notebook off-AMD).")

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda/rocm available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed in this kernel yet — pip install -r requirements.txt")

In [ ]:
# --- Install/ensure deps for the local server (safe to re-run) ---
%pip install -q fastapi uvicorn nest_asyncio requests python-dotenv

## Start Ollama and pull both models

This is what makes the "AMD Notebook — Qwen Coder" / "AMD Notebook — Gemma 2" provider
options in the frontend genuine on-GPU compute instead of a relay: both models are
pulled once here and served locally via Ollama's REST API on this instance. Qwen2.5-Coder
(not plain Llama 3) is used because every specialist agent asks the LLM to write Python
code that gets executed - a code-focused model is far more reliable at that than a
general chat model.

In [ ]:
import subprocess, time as _time

OLLAMA_MODELS = {"qwen2.5-coder:7b", "gemma2"}

subprocess.Popen(["ollama", "serve"])
_time.sleep(5)
for _model in OLLAMA_MODELS:
    pull = subprocess.run(["ollama", "pull", _model], capture_output=True, text=True)
    print(f"--- pull {_model} ---")
    print(pull.stdout[-300:] if pull.stdout else "(already present)")

In [ ]:
# --- Load the same FEATHERLESS_API_KEY the backend uses (for the fallback route) ---
# Put backend/.env (or a copy of it) next to this notebook, or export the var
# in the AMD Jupyter environment directly. Never hardcode the key here.
import os
from dotenv import load_dotenv

load_dotenv()  # looks for a .env in the notebook's working directory
load_dotenv("../backend/.env")  # also try the backend's .env if this notebook sits in amd_compute/

FEATHERLESS_API_KEY = os.environ.get("FEATHERLESS_API_KEY", "")
FEATHERLESS_MODEL = os.environ.get("FEATHERLESS_MODEL", "Qwen/Qwen2.5-7B-Instruct")
FEATHERLESS_URL = "https://api.featherless.ai/v1/chat/completions"

if FEATHERLESS_API_KEY:
    print(f"Loaded Featherless config. Model: {FEATHERLESS_MODEL}")
else:
    print("No FEATHERLESS_API_KEY found — that's fine if you're only using the llama3/gemma2 Ollama routes.")

In [ ]:
# --- Two call paths, executed from THIS notebook process on the AMD box ---
import time
import requests

def call_ollama_local(messages: list, model: str) -> str:
    """Runs `model` locally via Ollama's REST API on this AMD instance's GPU
    and returns the assistant's text content. This is the genuine on-device
    compute path used by the "AMD Notebook — Llama 3"/"Gemma 2" providers."""
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={"model": model, "messages": messages, "stream": False},
        timeout=90,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]

def call_featherless(messages: list, temperature: float = 0.2, max_tokens: int = 1000) -> dict:
    """Same retry-on-429/transient-error behavior as backend/agent_core.py's
    _post_with_retry, kept in sync manually since this now runs in a separate
    process/environment from the backend. Used for the Featherless fallback
    route (any request whose model isn't llama3/gemma2)."""
    if not FEATHERLESS_API_KEY:
        raise RuntimeError("FEATHERLESS_API_KEY not set in this notebook's environment.")
    last_exc = None
    for attempt in range(4):
        try:
            resp = requests.post(
                FEATHERLESS_URL,
                headers={
                    "Authorization": f"Bearer {FEATHERLESS_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": FEATHERLESS_MODEL,
                    "max_tokens": max_tokens,
                    "temperature": temperature,
                    "messages": messages,
                },
                timeout=30,
            )
        except requests.exceptions.RequestException as e:
            last_exc = e
            time.sleep(0.6 * (attempt + 1))
            continue
        if resp.status_code == 429:
            last_exc = RuntimeError(f"Rate limited (429), attempt {attempt + 1}/4")
            time.sleep(1.5 * (attempt + 1))
            continue
        resp.raise_for_status()
        return resp.json()
    raise last_exc if last_exc else RuntimeError("Request failed after retries.")

print("call_ollama_local() and call_featherless() ready.")

In [ ]:
# --- Local HTTP server: mirrors an OpenAI-style /v1/chat/completions shape ---
# backend/agent_core.py posts here. If body["model"] is qwen2.5-coder:7b/gemma2, this
# routes to the local Ollama call (genuine AMD on-GPU compute); otherwise it
# forwards to Featherless from this AMD notebook process (fallback route).
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

nest_asyncio.apply()

server_app = FastAPI(title="AMD Notebook Inference Relay")

@server_app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    messages = body.get("messages", [])
    model = body.get("model", FEATHERLESS_MODEL)
    temperature = body.get("temperature", 0.2)
    max_tokens = body.get("max_tokens", 1000)
    try:
        if model in OLLAMA_MODELS:
            content = call_ollama_local(messages, model)
            return JSONResponse({"choices": [{"message": {"role": "assistant", "content": content}}]})
        result = call_featherless(messages, temperature=temperature, max_tokens=max_tokens)
        return JSONResponse(result)
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=502)

@server_app.get("/health")
async def health():
    return {"status": "ok", "ollama_models": sorted(OLLAMA_MODELS), "featherless_model": FEATHERLESS_MODEL}

print("Server app defined. Run the next cell to start it (blocks the kernel while running).")

In [ ]:
# --- Start the server. Leave this cell running while backend/agent_core.py calls it. ---
# Bind 0.0.0.0 if the FastAPI backend runs on a different machine/container than this
# notebook and needs to reach it over the network; otherwise 127.0.0.1 is fine.
#
# NOTE: uvicorn.run() tries to own the event loop itself, which fails inside a
# notebook kernel (Jupyter already has a loop running). We build a Server object
# and await server.serve() instead -- that cooperates with the kernel's existing
# loop instead of fighting it for control. nest_asyncio.apply() above is a backup
# in case some other library later tries a raw uvicorn.run()/asyncio.run() again.
HOST = "0.0.0.0"
PORT = 8800
print(f"Starting AMD notebook inference relay on http://{HOST}:{PORT} ...")
config = uvicorn.Config(server_app, host=HOST, port=PORT, log_level="info")
server = uvicorn.Server(config)
await server.serve()